# NeoMint Planner -- Model Evaluation

Comprehensive offline evaluation suite for the fine-tuned **NeoMint-Planner-1.7B** model against held-out test and validation benchmarks.

### Evaluation Metrics:
1. **JSON Validity Rate**: Percentage of model responses that parse as valid JSON.
2. **Schema Compliance**: Conformance to `action-plan.schema.json` (`kind` in `[plan, clarification, refusal]`).
3. **Tool Grounding**: All proposed tools exist in `tool-manifest.json` and arguments match declared types.
4. **Policy Non-Contamination**: Zero occurrences of forbidden policy fields (`requires_confirmation`, `risk`, `permission`, `execute`).
5. **Intent & Category Accuracy**: Correct routing across read-only, state-changing, ambiguous, safety, and adversarial inputs.
6. **Inference Latency**: Latency and generation speed (tokens/sec) on T4 GPU.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers>=4.44.0 peft>=0.12.0 bitsandbytes>=0.43.0 accelerate jsonschema tabulate

## 2. Mount Google Drive or Upload Files

Upload `test.jsonl`, `validation.jsonl`, `tool-manifest.json`, and your saved LoRA adapter (`neomint-planner-1.7b-lora`).

In [ ]:
import os
import json
import time
from pathlib import Path

# Check if Google Drive is available, otherwise prompt file upload
use_drive = os.path.exists('/content/drive/MyDrive')
if not use_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        use_drive = True
    except Exception:
        print("Google Drive not mounted. Using local uploads.")

# If files aren't local, upload them
if not os.path.exists('test.jsonl') or not os.path.exists('tool-manifest.json'):
    from google.colab import files
    print("Upload test.jsonl, validation.jsonl, and tool-manifest.json:")
    uploaded = files.upload()

## 3. Configuration & Model Loading

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-1.7B"
# Point to local folder or Drive folder
ADAPTER_PATH = "neomint-planner-1.7b-lora"
if not os.path.exists(ADAPTER_PATH) and os.path.exists(f"/content/drive/MyDrive/neomint/{ADAPTER_PATH}"):
    ADAPTER_PATH = f"/content/drive/MyDrive/neomint/{ADAPTER_PATH}"

TEST_FILE = "test.jsonl"
MANIFEST_FILE = "tool-manifest.json"
OUTPUT_REPORT = "evaluation_report.json"

print(f"Base model: {BASE_MODEL}")
print(f"Adapter path: {ADAPTER_PATH}")

# 4-bit config for fast evaluation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if os.path.exists(ADAPTER_PATH):
    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
else:
    print(f"WARNING: Adapter path '{ADAPTER_PATH}' not found. Evaluating raw base model.")
    model = base_model

model.eval()
print("Model loaded successfully!")

## 4. Load Manifest and Rules

In [ ]:
with open(MANIFEST_FILE, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

MANIFEST_TOOLS = {t['name']: t for t in manifest.get('tools', [])}
FORBIDDEN_FIELDS = {'requires_confirmation', 'permission', 'risk', 'execute', 'risk_level'}
VALID_KINDS = {'plan', 'clarification', 'refusal'}

print(f"Loaded {len(MANIFEST_TOOLS)} valid tools from manifest:")
for name in sorted(MANIFEST_TOOLS.keys()):
    print(f"  - {name}")

## 5. Evaluation Harness & Scoring Functions

In [ ]:
def extract_assistant_response(full_output_text, prompt_length, tokenizer):
    """Extract generated assistant text without prompt and special tokens."""
    generated_tokens = full_output_text[prompt_length:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
    if "<|im_end|>" in text:
        text = text.split("<|im_end|>")[0]
    return text.strip()

def evaluate_prediction(pred_raw, target_obj, category):
    """Evaluate single prediction against schema and gold target."""
    result = {
        "is_valid_json": False,
        "is_valid_schema": False,
        "has_no_forbidden_fields": True,
        "tools_grounded": True,
        "kind_match": False,
        "exact_tool_match": False,
        "category": category,
        "errors": [],
        "parsed": None,
    }
    
    # 1. JSON parsing
    try:
        pred = json.loads(pred_raw)
        result["is_valid_json"] = True
        result["parsed"] = pred
    except Exception as e:
        result["errors"].append(f"JSON decode error: {e}")
        return result
    
    # 2. Forbidden policy fields
    def check_forbidden(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k in FORBIDDEN_FIELDS:
                    result["has_no_forbidden_fields"] = False
                    result["errors"].append(f"Forbidden policy field found: '{k}'")
                check_forbidden(v)
        elif isinstance(obj, list):
            for item in obj:
                check_forbidden(item)
    check_forbidden(pred)
    
    # 3. Schema kind
    kind = pred.get("kind")
    if kind not in VALID_KINDS:
        result["errors"].append(f"Invalid kind '{kind}', expected one of {VALID_KINDS}")
        return result
    
    target_kind = target_obj.get("kind")
    result["kind_match"] = (kind == target_kind)
    
    # 4. Kind-specific validation
    if kind == "plan":
        actions = pred.get("actions")
        if not isinstance(actions, list) or len(actions) == 0:
            result["errors"].append("Plan must contain non-empty 'actions' array")
        else:
            result["is_valid_schema"] = True
            pred_tools = []
            for idx, act in enumerate(actions):
                t_name = act.get("tool")
                pred_tools.append(t_name)
                if t_name not in MANIFEST_TOOLS:
                    result["tools_grounded"] = False
                    result["errors"].append(f"Ungrounded tool '{t_name}' in action {idx}")
                if not isinstance(act.get("arguments"), dict):
                    result["errors"].append(f"Action {idx} arguments must be an object")
            
            gold_tools = [a.get("tool") for a in target_obj.get("actions", [])]
            result["exact_tool_match"] = (pred_tools == gold_tools)

    elif kind == "clarification":
        if "question" in pred and "clarification_reason" in pred:
            result["is_valid_schema"] = True
        else:
            result["errors"].append("Clarification requires 'question' and 'clarification_reason'")
        result["exact_tool_match"] = (target_kind == "clarification")

    elif kind == "refusal":
        if "refusal_message" in pred and "refusal_reason" in pred:
            result["is_valid_schema"] = True
        else:
            result["errors"].append("Refusal requires 'refusal_message' and 'refusal_reason'")
        result["exact_tool_match"] = (target_kind == "refusal")
        
    return result

## 6. Run Test Evaluation Benchmark

In [ ]:
from tabulate import tabulate

test_samples = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            test_samples.append(json.loads(line))

print(f"Evaluating {len(test_samples)} test benchmark samples...")

detailed_results = []
start_eval_time = time.time()

for i, sample in enumerate(test_samples):
    sample_id = sample.get("id", f"sample_{i}")
    messages = sample["messages"]
    target = sample["target"]
    category = sample.get("labels", {}).get("category", "unknown")
    
    # Format chat prompt
    sys_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
    usr_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
    
    prompt_text = (
        f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
        f"<|im_start|>user\n{usr_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,  # Greedy decoding for strict evaluation
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    latency = time.time() - t0
    gen_tokens = outputs[0].shape[0] - prompt_len
    tok_per_sec = (gen_tokens / latency) if latency > 0 else 0.0
    
    raw_pred = extract_assistant_response(outputs[0], prompt_len, tokenizer)
    eval_res = evaluate_prediction(raw_pred, target, category)
    eval_res.update({
        "id": sample_id,
        "user_prompt": usr_msg,
        "gold_target": target,
        "model_raw": raw_pred,
        "latency_sec": round(latency, 3),
        "tokens_per_sec": round(tok_per_sec, 1),
    })
    detailed_results.append(eval_res)
    print(f"[{i+1}/{len(test_samples)}] {sample_id} ({category}) -> kind match: {eval_res['kind_match']} in {latency:.2f}s")

total_time = time.time() - start_eval_time
print(f"\nEvaluation completed in {total_time:.2f}s")

## 7. Compute Benchmark Metrics & Summary Table

In [ ]:
from collections import defaultdict

total_count = len(detailed_results)
valid_json_count = sum(1 for r in detailed_results if r["is_valid_json"])
valid_schema_count = sum(1 for r in detailed_results if r["is_valid_schema"])
forbidden_free_count = sum(1 for r in detailed_results if r["has_no_forbidden_fields"])
grounded_count = sum(1 for r in detailed_results if r["tools_grounded"])
kind_match_count = sum(1 for r in detailed_results if r["kind_match"])
tool_match_count = sum(1 for r in detailed_results if r["exact_tool_match"])
avg_tok_sec = sum(r["tokens_per_sec"] for r in detailed_results) / total_count if total_count else 0

# Per-category breakdown
category_stats = defaultdict(lambda: {"total": 0, "kind_ok": 0, "tool_ok": 0})
for r in detailed_results:
    cat = r["category"]
    category_stats[cat]["total"] += 1
    if r["kind_match"]:
        category_stats[cat]["kind_ok"] += 1
    if r["exact_tool_match"]:
        category_stats[cat]["tool_ok"] += 1

print("=" * 65)
print("         NEOMINT PLANNER EVALUATION REPORT")
print("=" * 65)
print(f"Total Test Samples:          {total_count}")
print(f"JSON Validity Rate:          {valid_json_count / total_count * 100:.1f}% ({valid_json_count}/{total_count})")
print(f"Schema Compliance Rate:      {valid_schema_count / total_count * 100:.1f}% ({valid_schema_count}/{total_count})")
print(f"Policy Non-Contamination:    {forbidden_free_count / total_count * 100:.1f}% ({forbidden_free_count}/{total_count})")
print(f"Tool Grounding Rate:         {grounded_count / total_count * 100:.1f}% ({grounded_count}/{total_count})")
print(f"Action Kind Accuracy:        {kind_match_count / total_count * 100:.1f}% ({kind_match_count}/{total_count})")
print(f"Tool Selection Accuracy:     {tool_match_count / total_count * 100:.1f}% ({tool_match_count}/{total_count})")
print(f"Average Generation Speed:    {avg_tok_sec:.1f} tokens/sec")
print("=" * 65)

cat_rows = []
for cat, s in sorted(category_stats.items()):
    k_pct = (s["kind_ok"] / s["total"]) * 100
    t_pct = (s["tool_ok"] / s["total"]) * 100
    cat_rows.append([cat, s["total"], f"{k_pct:.1f}%", f"{t_pct:.1f}%"])

print("\n--- Breakdown by Category ---")
print(tabulate(cat_rows, headers=["Category", "Total", "Kind Acc", "Tool Acc"], tablefmt="grid"))

## 8. Export Evaluation Report JSON

Save structured evaluation results for documentation and CI comparison.

In [ ]:
summary_report = {
    "model": BASE_MODEL,
    "adapter": ADAPTER_PATH,
    "total_samples": total_count,
    "metrics": {
        "json_validity_rate": valid_json_count / total_count if total_count else 0,
        "schema_compliance_rate": valid_schema_count / total_count if total_count else 0,
        "policy_non_contamination_rate": forbidden_free_count / total_count if total_count else 0,
        "tool_grounding_rate": grounded_count / total_count if total_count else 0,
        "kind_accuracy": kind_match_count / total_count if total_count else 0,
        "tool_selection_accuracy": tool_match_count / total_count if total_count else 0,
        "avg_tokens_per_sec": avg_tok_sec,
    },
    "by_category": dict(category_stats),
    "detailed_results": detailed_results,
}

with open(OUTPUT_REPORT, 'w', encoding='utf-8') as f:
    json.dump(summary_report, f, indent=2)

print(f"Full evaluation report saved to: {OUTPUT_REPORT}")
if use_drive:
    import shutil
    shutil.copy(OUTPUT_REPORT, '/content/drive/MyDrive/neomint/')
    print("Copied report to Google Drive /neomint/")